# BSP Registry Tools

### Streamlined Yocto & Isar BSP Management for Embedded Linux Teams

> *From registry to built image — one command*

---

📦 `pip install bsp-registry-tools`  
🔗 **Registry**: [github.com/Advantech-EECC/bsp-registry](https://github.com/Advantech-EECC/bsp-registry/tree/development)  
🔗 **Tools**: [github.com/Advantech-EECC/bsp-registry-tools](https://github.com/Advantech-EECC/bsp-registry-tools)  
⚖️ Apache 2.0 License


## The Problem

Yocto / KAS builds force teams to juggle many moving parts:

| Pain point | Without tooling |
|---|---|
| 30+ boards × 8 releases | Copy-pasted KAS config files |
| Docker environments per build | Undocumented, team-specific setup |
| Features (OTA, secure-boot, ROS 2, …) | Scattered patches, no reuse |
| Multiple SoC vendors (NXP, MediaTek, Qualcomm) | Separate ad-hoc scripts |
| Reproducibility | "Works on my machine" |
| Discoverability | "What boards do we support?" |

**One registry file → single source of truth for the whole team.**


## What is BSP Registry Tools?

- **Python CLI + library** for managing Board Support Packages
- **YAML registry** (v2.0 schema) as the single source of truth  
  — devices, releases, features, presets, environments, vendors
- Wraps **KAS** (`kas` / `kas-container`) to drive reproducible Yocto/Isar builds
- Works **out of the box** — auto-clones the Advantech registry on first run
- Targets: embedded Linux teams, BSP maintainers, CI/CD pipelines

```
bsp-registry-tools
├── CLI  (bsp build / list / shell / export / deploy / gather / registry / server …)
├── Python API  (BspManager, V2Resolver, RegistryFetcher, …)
└── HTTP server  (REST + GraphQL via FastAPI)
```


## Installation

### Core
```bash
pip install bsp-registry-tools
```

### Optional extras

| Extra | Installs |
|---|---|
| `[azure]` | Azure Blob Storage upload / download |
| `[aws]` | AWS S3 upload / download |
| `[server]` | FastAPI + uvicorn + Strawberry GraphQL |
| `[completions]` | Shell tab completions (argcomplete) |
| `[dev]` | pytest, coverage, ruff, … |

```bash
pip install "bsp-registry-tools[azure,completions]"
```


## Zero-Config Quick Start

No registry file needed — the tool auto-clones the Advantech registry (`development` branch) on first run.


In [ ]:
# First run: clones https://github.com/Advantech-EECC/bsp-registry → ~/.cache/bsp/registry
# Subsequent runs: git-pull to keep up to date
!bsp list


In [ ]:
# Skip network update (great for CI or offline use)
!bsp --no-update list


In [ ]:
# Target the development branch explicitly
!bsp --remote https://github.com/Advantech-EECC/bsp-registry.git --branch development list


## Registry Resolution Priority

The tool finds the registry in this order:

1. `--registry <path>` — explicit local file, no network access
2. `--local` — use `./bsp-registry.yml` in CWD, no network access
3. `./bsp-registry.yml` auto-detected in CWD
4. `./bsp-registry.yaml` auto-detected in CWD
5. **Remote clone** → `~/.cache/bsp/registry` (default: Advantech registry)

```bash
# Explicit local registry
bsp --registry /path/to/my-registry.yaml list

# Force local directory, never hit network
bsp --local list

# Use a named remote saved with `bsp remotes add`
bsp --remote advantech list
```


## Supported Hardware — NXP i.MX Boards

| Board | SoC | Releases | Status |
|---|---|---|---|
| **RSB-3720** | i.MX8M Plus | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **RSB-3720 4G** | i.MX8M Plus | walnascar, whinlatter | 🟢 Stable |
| **RSB-3720 6G** | i.MX8M Plus | walnascar, whinlatter | 🟢 Stable |
| **ROM-2620** | i.MX8 | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **ROM-2820** | i.MX93 | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **ROM-5720** | i.MX8 | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **ROM-5721** | i.MX8 | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **ROM-5722** | i.MX8 | scarthgap, styhead, walnascar, whinlatter | 🟢 Stable |
| **AOM-5521** | i.MX95 | scarthgap, walnascar | 🟢 Stable |

Plus **MediaTek** (RSB-3810, Genio 1200 EVK) and **Qualcomm** (AOM-2721, QCS6490 RB3gen2) targets.

**Emulated targets**: `qemuarm64`, `qemuarm`, `qemux86`, `qemux86-64`, `qemuamd64`


## Supported Yocto Releases

| Slug | Yocto | LTS | Distros available |
|---|---|---|---|
| `kirkstone` | 4.0 | ✅ LTS | poky, fsl-imx-xwayland |
| `mickledore` | 4.2 | | poky, fsl-imx-xwayland |
| `nanbield` | 4.3 | | poky |
| `scarthgap` | 5.0 | ✅ LTS | poky, fsl-imx-xwayland, mediatek, qualcomm, ros2 |
| `styhead` | 5.1 | | poky, fsl-imx-xwayland |
| `walnascar` | 5.2 | | poky, fsl-imx-xwayland |
| `whinlatter` | 5.3 | | poky, fsl-imx-xwayland |
| `wrynose` | 6.0 | | poky |

**Isar (Debian-based) releases:**

| Slug | Distribution |
|---|---|
| `debian-trixie` | Debian 13 (Trixie) |
| `ubuntu-noble` | Ubuntu 24.04 LTS |
| `ubuntu-jammy` | Ubuntu 22.04 LTS |

Many presets support **multiple releases** via the `releases: [...]` field — one preset, many targets.


## Registry Schema v2 — Separation of Concerns

v2 decomposes the registry into independent, reusable sections:

| Section | Purpose |
|---|---|
| `frameworks` | Build-system framework definitions (Yocto, Isar) |
| `distro` | Distribution definitions (Poky, fsl-imx-xwayland, Isar v1.0 …) |
| `vendors` | Cross-release board-vendor KAS fragments (Advantech, NXP, MediaTek …) |
| `devices` | Hardware board definitions (slug, vendor, SoC, KAS includes) |
| `releases` | Yocto / Isar release definitions with vendor overrides |
| `features` | Optional add-ons (OTA, secure-boot, ROS 2, hailo …) |
| `bsp` | Named presets = device + release(s) + features |
| `environments` | Container + variable bundles per build class |
| `containers` | Docker image definitions |
| `deploy` | Cloud artifact upload configuration (Azure / AWS) |
| `include` | Split large registries across files |

**Builds can be driven by a named preset _or_ by composing components directly:**

```bash
bsp build modular-bsp-rsb3720             # named preset (latest release auto-selected)
bsp build --device rsb3720 --release scarthgap  # component-based
```


## Containers & Named Environments

### Containers (real definitions from the registry)

```yaml
containers:
  - ubuntu-22.04:
      file: Dockerfile.ubuntu
      image: "advantech/bsp-registry/ubuntu-22.04/kas:5.2"
      args:
        - { name: DISTRO,      value: "ubuntu:22.04" }
        - { name: KAS_VERSION, value: "5.2" }

  - ubuntu-22.04-csb:          # Secure-Boot build: mounts signing keys
      file: Dockerfile.ubuntu
      image: "advantech/bsp-registry/ubuntu-22.04/kas:5.2"
      volumes:
        - { host: "$ENV{CST_TOOL_PATH}", container: "/opt/cst/" }
        - { host: "$ENV{KEYS_PATH}",     container: "/opt/keys/" }

  - isar-debian-13:            # Isar builds need privileged mode
      file: Dockerfile.isar.debian
      image: "advantech/bsp-registry/isar/debian-13/kas:5.2"
      runtime_args: "-p 2222:2222 --device=/dev/net/tun --cap-add=NET_ADMIN"
      privileged: true
```

### Named environments

```yaml
environments:
  default:            { container: ubuntu-22.04 }
  ubuntu-22.04-csb:   { container: ubuntu-22.04-csb,
                        variables: [{name: SIG_TOOL_PATH, value: "/opt/cst"}] }
  isar-build-environment:
    container: isar-debian-13
    copy: [{ "isar/scripts/isar-runqemu.sh": "build/" }]
    variables:
      - { name: DL_DIR,    value: "$ENV{HOME}/data/cache/isar/downloads" }
      - { name: SSTATE_DIR, value: "$ENV{HOME}/data/cache/isar/sstate" }
```


## Registry v2 — Devices

```yaml
registry:
  devices:
    # Emulated targets
    - slug: qemuarm64
      description: "QEMU ARM64 (emulated)"
      vendor: qemu
      soc_vendor: arm
      architecture: aarch64
      includes: [vendors/qemu/machine/qemuarm64.yaml]

    # Advantech Europe NXP i.MX8 boards
    - slug: rsb3720
      description: "Advantech RSB-3720 (i.MX8, 6GB)"
      vendor: advantech-europe
      soc_vendor: nxp
      architecture: arm64
      includes: [vendors/advantech-europe/nxp/machine/imx8/rsb3720.yml]

    - slug: rom2820-ed93
      description: "Advantech ROM-2820 (i.MX93)"
      vendor: advantech-europe
      soc_vendor: nxp
      architecture: arm64
      includes: [vendors/advantech-europe/nxp/machine/imx9/rom2820-ed93.yml]

    # Advantech MediaTek / Qualcomm
    - slug: rsb3810
      description: "Advantech RSB-3810 (Mediatek)"
      vendor: advantech-europe
      soc_vendor: mediatek
      architecture: arm64
      includes: [vendors/advantech-europe/mediatek/machine/rsb3810.yaml]
```


## Registry v2 — Named Presets (BSP)

```yaml
registry:
  bsp:
    # QEMU — covers ALL Yocto releases in one preset
    - name: poky-qemuarm64
      description: "Poky QEMU ARM64 (Yocto 5.0 LTS)"
      device: qemuarm64
      releases: [kirkstone, nanbield, mickledore, scarthgap, styhead, walnascar, whinlatter]
      features: [systemd, usrmerge, yocto-ssh]
      build: { path: build/poky-qemuarm64 }

    # RSB-3720 — standard Yocto build
    - name: modular-bsp-rsb3720
      description: "Advantech RSB-3720 (i.MX8)"
      device: rsb3720
      releases: [scarthgap, styhead, walnascar, whinlatter]
      features: [systemd, security, virtualization, ipv6, usrmerge]
      build: { path: build/modular-bsp-rsb3720 }

    # RSB-3720 with RAUC OTA
    - name: modular-bsp-rauc-rsb3720
      description: "Advantech RSB-3720 (i.MX8) with RAUC OTA"
      device: rsb3720
      releases: [scarthgap, styhead, walnascar, whinlatter]
      features: [systemd, security, virtualization, ipv6, usrmerge, rauc]
      build: { path: build/modular-bsp-rauc-rsb3720 }

    # Isar — QEMU AMD64 across multiple Isar distros
    - name: isar-qemuamd64
      description: "Isar QEMU AMD64"
      device: qemuamd64
      releases: [ubuntu-noble, ubuntu-jammy, debian-trixie]
      features: []
      build: { path: build/isar-qemuamd64 }
```

```bash
bsp build modular-bsp-rsb3720                  # uses default/first release
bsp build modular-bsp-rsb3720 --release walnascar  # specific release
```


## Feature System — Real Feature Catalogue

| Slug | Description | Framework |
|---|---|---|
| `systemd` | Enable systemd as init system | Yocto |
| `yocto-ssh` | SSH server in the image | Yocto |
| `debug-tweaks` | Debug build tweaks | Yocto |
| `root-login` | Enable root login | Yocto |
| `security` | Security hardening | Yocto |
| `secure-boot` | NXP HAB / AHAB signing (i.MX8 / i.MX9 / i.MX95) | Yocto |
| `virtualization` | KVM / container support | Yocto |
| `wayland` | Wayland display support | Yocto |
| `x11` | X11 display support | Yocto |
| `ipv6` | IPv6 networking | Yocto |
| `udev` | udev device manager | Yocto |
| `usrmerge` | /usr merge | Yocto |
| **`rauc`** | RAUC OTA (Robust Auto-Update Controller) | Yocto |
| **`swupdate`** | SWUpdate OTA | Yocto |
| **`ostree`** | OSTree atomic upgrades | Yocto |
| `hailo` | Hailo-8 AI accelerator support | Yocto |
| `ros2` | ROS 2 (Robot Operating System 2) — via `ros2-humble-scarthgap` release | Yocto |

```bash
# Build RSB-3720 with RAUC OTA
bsp build --device rsb3720 --release walnascar --features systemd,security,rauc

# Build RSB-3721 with Secure Boot
bsp build modular-bsp-rom5721-2g-db5901-secureboot
```


## Vendor Overrides — Multi-SoC / Multi-BSP Releases

The real `scarthgap` release shows the full power of vendor overrides:

```yaml
releases:
  - slug: scarthgap
    distro: poky
    description: "Yocto 5.0 LTS (Scarthgap)"
    yocto_version: "5.0"
    includes: [compilers/clang/clang.yml, yocto/releases/scarthgap.yml]
    vendor_overrides:
      - vendor: advantech-europe
        soc_vendors:
          - vendor: nxp
            distro: fsl-imx-xwayland   # replaces poky for all Advantech-Europe NXP boards
            includes: [vendors/advantech-europe/nxp/modular-bsp-nxp.yml]
            releases:
              - slug: imx-6.6.52-2.2.2    # Kernel 6.6.52
                includes: [vendors/advantech-europe/nxp/imx-6.6.52-2.2.2-scarthgap.yml]
              - slug: imx-6.6.52-2.2.0    # Kernel 6.6.52 (earlier BSP)
                includes: [vendors/advantech-europe/nxp/imx-6.6.52-2.2.0-scarthgap.yml]
          - vendor: mediatek             # MediaTek boards in the same release
            includes: [vendors/advantech-europe/mediatek/modular-bsp-mediatek.yml]
            releases:
              - slug: mtk-rity-v25.0
                includes: [vendors/advantech-europe/mediatek/mtk-rity-v25.0-scarthgap.yml]
      - vendor: qualcomm                 # Qualcomm boards in the same release
        releases:
          - slug: qcs6490-rb3gen2-vision-kit
            includes: [vendors/qualcomm/qcom-6.6.97-qli.1.6-ver.1.2-scarthgap.yml]
```

> 📌 One `scarthgap` release serves NXP i.MX8, MediaTek Rity, and Qualcomm QLI — zero duplication.


## Isar BSPs — Debian-Based Embedded Linux

Isar uses Debian-native tooling (apt / dpkg) instead of BitBake:

```yaml
environments:
  isar-build-environment:
    container: isar-debian-13
    copy: [{ "isar/scripts/isar-runqemu.sh": "build/" }]

registry:
  bsp:
    - name: isar-qemuamd64
      device: qemuamd64
      releases: [ubuntu-noble, ubuntu-jammy, debian-trixie]
      features: []
      build: { path: build/isar-qemuamd64 }

    - name: isar-qemuarm64
      device: qemuarm64
      releases: [debian-trixie]
      features: []
      build: { path: build/isar-qemuarm64 }
```


In [ ]:
# Build QEMU AMD64 with Ubuntu Noble via Isar
!bsp build isar-qemuamd64 --release ubuntu-noble

# Build QEMU AMD64 with Debian Trixie via Isar
!bsp build isar-qemuamd64 --release debian-trixie

# Build Advantech RSB-3720 with Isar (development target)
!bsp build adv-mbsp-isar-debian-rsb3720


## OTA Update Support

Three OTA technologies supported — all as composable features:

| Technology | Slug | Supported boards |
|---|---|---|
| **RAUC** | `rauc` | RSB-3720, ROM-2620, ROM-2820, ROM-5720, ROM-5721, ROM-5722 |
| **SWUpdate** | `swupdate` | RSB-3720, ROM-2620, ROM-2820, ROM-5720, ROM-5721, ROM-5722 |
| **OSTree** | `ostree` | RSB-3720, ROM-2620, ROM-2820, ROM-5720, ROM-5721, ROM-5722 |

OSTree includes release-specific patches automatically:

```yaml
features:
  - slug: ostree
    compatible_with: [yocto]
    includes: [features/ota/ostree/ostree.yml]
    release_overrides:           # release-specific patches, auto-applied
      - { release: scarthgap, includes: [features/ota/ostree/ostree-scarthgap.yml] }
      - { release: styhead,   includes: [features/ota/ostree/ostree-styhead.yml] }
      - { release: walnascar, includes: [features/ota/ostree/ostree-walnascar.yml] }
```


In [ ]:
# List all OTA-related presets
!bsp list | grep -E 'rauc|swupdate|ostree'

# Build RSB-3720 with RAUC (all supported releases available)
!bsp build modular-bsp-rauc-rsb3720 --release walnascar

# Build RSB-3720 with SWUpdate
!bsp build modular-bsp-swupdate-rsb3720 --release scarthgap

# Build RSB-3720 with OSTree
!bsp build modular-bsp-ostree-rsb3720 --release walnascar


## Secure Boot — NXP HAB & AHAB

| SoC Family | Technology | Boards |
|---|---|---|
| i.MX8 | **HAB** (High Assurance Boot) | RSB-3720, ROM-2620, ROM-5720, ROM-5721, ROM-5722 |
| i.MX93 | **AHAB** (Advanced HAB) | ROM-2820 |
| i.MX95 | **AHAB** | AOM-5521 |

```yaml
features:
  - slug: secure-boot
    compatible_with: [yocto]
    includes: [features/secure-boot/secure-boot.yml]
    vendor_overrides:
      - vendor: advantech-europe
        soc_vendors:
          - vendor: nxp
            includes: [vendors/nxp/features/secure-boot/imx-secure-boot.yml]
```


In [ ]:
# Provide signing keys via environment variables
import os
os.environ.update({
    "CST_TOOL_PATH": "/opt/nxp/cst/",        # NXP CST tool path on host
    "KEYS_PATH":     "/path/to/srk-keys/",    # SRK keys directory on host
})

# Build ROM-5721 2G with Secure Boot (uses ubuntu-22.04-csb container with key mounts)
!bsp build modular-bsp-rom5721-2g-db5901-secureboot --release walnascar

# For ad-hoc builds add the feature directly
!bsp build --device rsb3720 --release walnascar --features systemd,security,secure-boot


## Core CLI — List & Discover


In [ ]:
# List all named presets (real output from the Advantech registry)
!bsp list


In [ ]:
# List devices
!bsp list devices


In [ ]:
# List releases
!bsp list releases


In [ ]:
# List available optional features
!bsp list features


In [ ]:
# List container definitions
!bsp containers


## Core CLI — Build


In [ ]:
# Build a named preset (uses default/first release)
!bsp build modular-bsp-rsb3720


In [ ]:
# Target a specific release from the preset's release list
!bsp build modular-bsp-rsb3720 --release walnascar


In [ ]:
# Build by composing components directly (no preset required)
!bsp build --device rsb3720 --release scarthgap


In [ ]:
# Checkout / validate only — fast, no actual build (CI-friendly)
!bsp build modular-bsp-rsb3720 --checkout


In [ ]:
# Override build output directory
!bsp build modular-bsp-rsb3720 --path /mnt/fast-ssd/build


In [ ]:
# Clean build dir first, then build
!bsp build modular-bsp-rsb3720 --clean


## Core CLI — Shell & Export


In [ ]:
# Interactive shell inside the build container
# !bsp shell modular-bsp-rsb3720

# Execute a single command without entering an interactive shell
!bsp shell modular-bsp-rsb3720 --command "bitbake -e core-image-minimal | grep ^MACHINE=" 


In [ ]:
# Export the resolved KAS configuration to stdout
!bsp export modular-bsp-rsb3720 --release scarthgap


In [ ]:
# Save the exported config to a file for archiving or sharing
!bsp export modular-bsp-rsb3720 --release walnascar --output /tmp/rsb3720-walnascar.yaml
!cat /tmp/rsb3720-walnascar.yaml


## Extending BSPs with Custom Yocto Layers

Any BSP preset's KAS config can be included in your own project:

```yaml
# my-custom-kas.yaml
header:
  version: 19
  includes:
    - repo: bsp-registry
      file: modular-bsp-rsb3720-walnascar.yaml   # export from `bsp export`

repos:
  bsp-registry:
    url: "https://github.com/Advantech-EECC/bsp-registry"
    branch: "development"
    layers: { .: "disabled" }

  meta-custom:
    layers:
      meta-custom:        # your custom layer
```

**Custom layer structure:**

```
meta-custom/
├── .config.yaml
└── meta-custom/
    ├── conf/layer.conf
    └── recipes-core/
        └── imx-image-%.bbappend   # CORE_IMAGE_EXTRA_INSTALL += "mpv"
```

```bash
kas build my-custom-kas.yaml
```


## Remote Registries & Remotes Management

Save frequently-used registry URLs as named remotes:


In [ ]:
# Add the Advantech development registry as a named remote
!bsp remotes add advantech https://github.com/Advantech-EECC/bsp-registry.git@development
# Add your own org registry
!bsp remotes add my-org https://github.com/my-org/bsp-registry.git


In [ ]:
# Show all saved remotes
!bsp remotes show


In [ ]:
# Use a saved remote by name
!bsp --remote advantech list

# Or by full URL@branch
!bsp --remote https://github.com/Advantech-EECC/bsp-registry.git@development list


In [ ]:
# Rename / update a remote
!bsp remotes rename my-org production
!bsp remotes set-url production https://github.com/my-org/bsp-registry-prod.git

# Remove a remote
!bsp remotes remove production


## Multi-Registry Mode

Load several independent registries simultaneously:

```bash
# Combine the upstream Advantech registry with your own additions
bsp --registry /path/to/advantech-registry.yaml --registry my-additions.yaml list

# Disambiguate with registry:preset syntax
bsp build advantech:modular-bsp-rsb3720
bsp build my-additions:custom-board-preset
```

```python
from bsp import BspManager

manager = BspManager(
    config_paths=[
        ("advantech",    "/path/to/advantech-registry.yaml"),
        ("my-additions", "/path/to/my-registry.yaml"),
    ]
)
manager.initialize()
manager.list_bsps()   # annotates output with [advantech] and [my-additions] prefixes
```


## Registry Management CLI — `bsp registry`

Full CRUD on every registry entity, backed by `RegistryWriter` (atomic saves, undo stack, git helpers).


In [ ]:
# Scaffold a new registry file
!bsp registry init --output /tmp/new-registry.yaml
!cat /tmp/new-registry.yaml


In [ ]:
# Validate an existing registry file
!bsp --local registry validate


In [ ]:
# Add a new device entry
!bsp --registry /tmp/new-registry.yaml registry add device \
    --slug rsb3720 \
    --description "Advantech RSB-3720 (i.MX8, 6GB)" \
    --vendor advantech-europe \
    --soc-vendor nxp \
    --includes vendors/advantech-europe/nxp/machine/imx8/rsb3720.yml


In [ ]:
# Compare two registry versions (unified diff)
!bsp registry diff /tmp/registry-v1.yaml /tmp/registry-v2.yaml


## Cloud Artifact Deployment

Upload Yocto build outputs to **Azure Blob Storage** or **AWS S3** after a build.

### Real deployment configuration (from bsp-registry.yml)

```yaml
deploy:
  provider: azure
  account_url: $ENV{AZURE_STORAGE_ACCOUNT_URL}
  container: bsp-registry-artifacts
  prefix: "{vendor}/{device}/{release}/{date}"
  patterns:
    - "**/*.wic.gz"
    - "**/*.wic.bz2"
    - "**/*.tar.bz2"
    - "**/*.ext4"
    - "**/*.img"
    - "**/*.bin"
    - "**/bzImage"
  artifact_dirs:
    - build/tmp/deploy/images
    - build/tmp/deploy/sdk
  include_manifest: true     # uploads SHA-256 JSON manifest
```


In [ ]:
# Deploy artifacts after an existing build
!bsp deploy modular-bsp-rsb3720 --release walnascar

# Build + deploy in one step
!bsp build modular-bsp-rsb3720 --release walnascar --deploy

# Dry-run: print what would be uploaded without uploading
!bsp deploy modular-bsp-rsb3720 --dry-run


## Artifact Gathering (Download)

Mirror of `bsp deploy` — download previously uploaded artifacts from cloud storage.


In [ ]:
# Download artifacts for a named preset
!bsp gather modular-bsp-rsb3720 --release walnascar --output /tmp/artifacts/

# Gather by components (no preset required)
!bsp gather --device rsb3720 --release walnascar --output /tmp/artifacts/


### Python API

```python
from bsp import BspManager

manager = BspManager("bsp-registry.yml")
manager.initialize()

result = manager.gather_bsp(
    preset_name="modular-bsp-rsb3720",
    output_dir="/tmp/artifacts",
)
for artifact in result.downloaded:
    print(artifact.name, artifact.size)
```


## HTTP Server — REST + GraphQL

```bash
pip install "bsp-registry-tools[server]"
```


In [ ]:
import subprocess, time
srv = subprocess.Popen(["bsp", "server", "--port", "8080"])
time.sleep(2)


In [ ]:
import requests

# REST — list all devices
r = requests.get("http://127.0.0.1:8080/api/v1/devices")
r.json()


In [ ]:
# GraphQL — query releases
query = """
{ releases { slug description yoctoVersion } }
"""
r = requests.post("http://127.0.0.1:8080/graphql", json={"query": query})
r.json()


In [ ]:
srv.terminate()


## Shell Tab Completions

Context-aware completions for every CLI argument:

```bash
pip install "bsp-registry-tools[completions]"

bsp completions bash >> ~/.bashrc && source ~/.bashrc
bsp completions zsh  >> ~/.zshrc  && source ~/.zshrc
bsp completions fish > ~/.config/fish/completions/bsp.fish
```

| Completer | Completes |
|---|---|
| `PresetsCompleter` | All named BSP presets — `modular-bsp-rsb3720`, `isar-qemuamd64`, … |
| `DevicesCompleter` | All device slugs — `rsb3720`, `rom2620-ed91`, `qemuarm64`, … |
| `ReleasesCompleter` | All release slugs — `scarthgap`, `walnascar`, `debian-trixie`, … |
| `FeaturesCompleter` | All feature slugs — `rauc`, `secure-boot`, `ostree`, … |
| `RemotesCompleter` | All saved remote names |

```bash
bsp build <TAB>                   # → modular-bsp-rsb3720  isar-qemuamd64  …
bsp build --device <TAB>          # → rsb3720  rom2620-ed91  qemuarm64  …
bsp build --release <TAB>         # → scarthgap  walnascar  debian-trixie  …
bsp build --features <TAB>        # → rauc  swupdate  ostree  secure-boot  …
```


## Python API


In [ ]:
from bsp import BspManager, RegistryFetcher

# ── 1. Fetch / update the Advantech development registry ──────────────────
fetcher = RegistryFetcher()
registry_path = fetcher.fetch_registry(
    repo_url="https://github.com/Advantech-EECC/bsp-registry.git",
    branch="development",
    update=True,
)
print("Registry path:", registry_path)


In [ ]:
# ── 2. Load registry & inspect contents ───────────────────────────────────
manager = BspManager(str(registry_path))
manager.initialize()

print("Devices:")
for dev in manager.model.registry.devices:
    print(f"  {dev.slug}: {dev.description} (vendor={dev.vendor}, soc={dev.soc_vendor})")

print("\nReleases:")
for rel in manager.model.registry.releases:
    v = getattr(rel, 'yocto_version', None)
    print(f"  {rel.slug}: {rel.description}" + (f" [Yocto {v}]" if v else ""))


In [ ]:
# ── 3. Resolve a preset into a full build configuration ───────────────────
from bsp import V2Resolver

v2_resolver = V2Resolver(manager.model)
config = v2_resolver.resolve(
    device_slug="rsb3720",
    release_slug="walnascar",
    feature_slugs=["systemd", "security", "rauc"],
)

print("KAS files in resolution order:")
for f in config.kas_files:
    print(" ", f)


## Python API — EnvironmentManager & RegistryWriter


In [ ]:
import os
from bsp import EnvironmentManager, EnvironmentVariable

# $ENV{} expansion — same as in the registry YAML
env_vars = [
    EnvironmentVariable(name="DL_DIR",     value="$ENV{HOME}/data/cache/downloads"),
    EnvironmentVariable(name="SSTATE_DIR", value="$ENV{HOME}/data/cache/sstate"),
    EnvironmentVariable(name="GITCONFIG_FILE", value="$ENV{HOME}/.gitconfig"),
]
env = EnvironmentManager(env_vars)
print("DL_DIR     →", env.get_value("DL_DIR"))
print("SSTATE_DIR →", env.get_value("SSTATE_DIR"))


In [ ]:
# RegistryWriter: programmatic CRUD + validation + git helpers
# from bsp.registry_writer import RegistryWriter
#
# writer = RegistryWriter("bsp-registry.yml")
# writer.add_device(slug="my-custom-board", description="My Board",
#                   vendor="my-vendor", soc_vendor="nxp",
#                   includes=["vendors/my-vendor/machine/my-board.yaml"])
# issues = writer.validate()
# if not issues:
#     writer.save()           # atomic write with backup
#     writer.git_stage()      # git add
#     writer.git_commit("Add my-custom-board device")
print("RegistryWriter: add/edit/remove + validate + undo + diff + git helpers")


## CI/CD Integration

```yaml
# .github/workflows/build.yml
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Install bsp-registry-tools
        run: pip install "bsp-registry-tools[azure]"

      - name: Validate registry
        run: bsp --local registry validate

      - name: Checkout BSP config (fast gate — no full build)
        run: bsp --no-update build modular-bsp-rsb3720 --checkout

      - name: Build & deploy artifacts to Azure
        env:
          AZURE_STORAGE_ACCOUNT_URL: ${{ secrets.AZURE_STORAGE_ACCOUNT_URL }}
        run: |
          bsp --no-update build modular-bsp-rsb3720 --release walnascar --deploy
```

| CI pattern | Command |
|---|---|
| Gate on registry validity | `bsp registry validate` |
| Fast checkout (no build) | `bsp build <preset> --checkout` |
| Deterministic, no network | `bsp --no-update` |
| Build + upload in one step | `bsp build <preset> --deploy` |
| Download from storage | `bsp gather <preset>` |


## Architecture

```
┌──────────────────────────────────────────────────────────┐
│  CLI  (bsp/cli.py)                                       │
│  HTTP Server  (bsp/server/ — FastAPI + Strawberry)       │
└─────────────────────┬────────────────────────────────────┘
                      │
┌─────────────────────▼────────────────────────────────────┐
│  BspManager  (bsp/bsp_manager.py)                        │
│    • Coordinates all operations                          │
│    • Multi-registry support                              │
└──┬──────────┬──────────┬─────────────┬───────────────────┘
   │          │          │             │
   ▼          ▼          ▼             ▼
V2Resolver  KasManager  ArtifactDeployer  ArtifactGatherer
   │                    │                  │
   │              Azure / AWS Storage Backends
   │
   ▼
RegistryFetcher   RemotesManager   RegistryWriter
```

**Data layer:** `dacite`-based dataclasses — `RegistryRoot`, `Device`, `Release`, `Feature`,  
`BspPreset`, `Framework`, `Distro`, `Vendor`, `NamedEnvironment`, `DeployConfig`, …


## Summary

| Feature | Benefit |
|---|---|
| 📋 YAML registry (v2) | Single source of truth for 30+ boards × 8+ releases |
| 🌐 Auto remote fetch | Zero-config first run; teams share the Advantech registry |
| 🏗️ Two build systems | Yocto/BitBake **and** Isar/Debian in one registry |
| 🔧 KAS integration | Reproducible builds for every device × release combo |
| 🐳 Named environments | Correct container per build class (secure-boot, Isar, …) |
| ✨ Feature system | Composable add-ons (OTA, secure-boot, ROS 2, Hailo) |
| 🔄 OTA support | RAUC, SWUpdate, OSTree — all as first-class features |
| 🔒 Secure Boot | NXP HAB / AHAB signing built into the registry |
| ☁️ Cloud deploy/gather | Full artifact lifecycle — build → upload → download |
| 🖥️ HTTP server | REST + GraphQL access for dashboards and automation |
| 🔡 Tab completions | Fast daily CLI use with context-aware completions |
| 🐍 Python API | Integrate into custom tools and CI scripts |

---

```bash
pip install bsp-registry-tools
bsp list   # auto-clones the Advantech development registry
```

🔗 [github.com/Advantech-EECC/bsp-registry](https://github.com/Advantech-EECC/bsp-registry/tree/development)
